In [1]:
!pip --version


pip 26.2.1 from D:\shproject\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_teddynote.models import MultiModal
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI

In [4]:
# 멀티모달 (이미지, 오디오 등도 받기)

import base64
from pathlib import Path

from langchain_core.messages import HumanMessage, SystemMessage

In [5]:
# 객체 생성

llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-5-nano",
)

In [6]:
# 이미지 주소, 파일 경로 받기
# 인터넷 이미지 주소는 변환하지 않고 그대로 사용
def image_url(image_source):
    if str(image_source).startswith(("http://", "https://")):
        return str(image_source)

    # 경로를 바꿔서 다루기 쉽게 만들고, PNG/JPG인지 확인
    image_path = Path(image_source)
    mimetype = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"

    # 이미지 파일 자체를 읽어서 긴 문자열 형태로 변환
    encoded_image = base64.b64encode(
        image_path.read_bytes()
    ).decode("utf-8")

    # 모델이 받을 수 있는 data URL 형태로 반환
    return f"data:{mimetype};base64,{encoded_image}"

In [7]:
# 입력 값 3개, 이미지 경로나 URL, 사용자가 물어볼 질문, 모델에게 주는 지침
def stream_image_response(image_source, user_prompt, system_prompt=None):
    # 텍스트 질문 + 이미지를 한 번에 묶는 파트
    content = [
        {"type": "text", "text": user_prompt},
        {"type": "image_url", "image_url": {"url": image_url(image_source)}}
    ]
    messages = [HumanMessage(content=content)]

    if system_prompt:
        messages.insert(0, SystemMessage(content=system_prompt))

    # 스트리밍 출력
    for token in llm.stream(messages):
        print(token.content, end="", flush=True)

    print()         

In [8]:
# 멀티모달 객체 생성
multimodal_llm = MultiModal(llm)

In [ ]:
# 이미지 제시
IMAGE_URL = "https://t3.ftcdn.net/jpg/03/77/33/96/360_F_377339633_Rtv9I77sSmSNcev8bEcnVxTHrXB4nRJ5.jpg"

# 이미지 파일로 부터 질의
# answer = multimodal_llm.stream(IMAGE_URL)
# stream_response(answer)

# 이미지 파일로 부터 질의
stream_image_response(
    IMAGE_URL,
    "이미지를 설명해 주세요.",
)

In [ ]:
# 시스템 프롬프트, 사용자 프롬프트 지정하기

system_prompt = """당신은 표(재무제표) 를 해석하는 금융 AI 어시스턴트 입니다.
당신의 임무는 주어진 테이블 형식의 재무제표를 바탕으로 흥미로운 사실을 정리하여 친절하게 답변하는 것입니다."""

user_prompt = """당신에게 주어진 표는 회사의 재무제표 입니다. 흥미로운 사실을 정리하여 답변하세요."""